# 01 · Exploratory data analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bileldh-ing/PrintGuard-/blob/main/notebooks/01_eda.ipynb)

Looks at the processed PrintGuard datasets built by `make data`:

* **Trier FFF** (Bast et al. 2026, Zenodo 10.5281/zenodo.18628777) – image-level classes per camera
* **YOLO-Strings** (Charia et al. 2025, Zenodo 14711320) – stringing boxes and per-print sequences

Questions: class balance per split and camera, image sizes, box sizes, sequence lengths, and a visual
sample grid. Runs on CPU; set `PROFILE = "full"` for the complete datasets.

In [ ]:
# Clone the repo (skipped when the notebook already runs inside it)
import os, subprocess, sys
if not os.path.exists("printguard/__init__.py"):
    if not os.path.exists("PrintGuard-"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bileldh-ing/PrintGuard-.git"], check=True)
    os.chdir("PrintGuard-")
print(os.getcwd())

In [ ]:
PROFILE = "smoke"   # "smoke" (few hundred MB) or "full" (tens of GB)
!pip -q install -r requirements.txt matplotlib pandas
if not os.path.exists("data/processed/splits.csv"):
    !make data PROFILE={PROFILE}

## Splits overview

In [ ]:
import pandas as pd
splits = pd.read_csv("data/processed/splits.csv")
print(splits.shape)
display(splits.head())
display(splits.groupby(["dataset", "split"]).size().unstack(fill_value=0))

Group-wise splits: every group (print block / recording sequence) must sit in exactly one split.

In [ ]:
leaks = splits.groupby(["dataset", "group"])["split"].nunique()
print("groups in more than one split:", int((leaks > 1).sum()))

## Trier FFF – class balance per split and camera

In [ ]:
from pathlib import Path
rows = []
for p in Path("data/processed/trier_cls").glob("*/*/*.jpg"):
    split, cls = p.parts[-3], p.parts[-2]
    rows.append({"split": split, "class": cls, "camera": p.name.split("__")[0]})
trier = pd.DataFrame(rows)
display(pd.crosstab([trier["split"], trier["camera"]], trier["class"]))

In [ ]:
import matplotlib.pyplot as plt
ax = pd.crosstab(trier["split"], trier["class"]).plot.bar(figsize=(7, 3.5), rot=0)
ax.set_ylabel("images"); ax.set_title("Trier FFF: images per class and split"); plt.tight_layout()

## Image sizes

In [ ]:
import cv2, random
random.seed(0)
sample = random.sample(list(Path("data/processed").rglob("*.jpg")), k=min(300, sum(1 for _ in Path("data/processed").rglob("*.jpg"))))
sizes = pd.DataFrame([cv2.imread(str(p)).shape[:2] for p in sample], columns=["h", "w"])
display(sizes.describe())

## YOLO-Strings – boxes per image and box size

In [ ]:
import numpy as np
lab = []
for f in Path("data/processed/yolo_strings/labels").glob("*/*.txt"):
    split = f.parent.name
    lines = [l.split() for l in f.read_text().splitlines() if l.strip()]
    lab.append({"split": split, "boxes": len(lines),
                "mean_area": float(np.mean([float(l[3]) * float(l[4]) for l in lines])) if lines else 0.0})
lab = pd.DataFrame(lab)
display(lab.groupby("split")[["boxes", "mean_area"]].describe().round(3))
lab["boxes"].plot.hist(bins=20, figsize=(6, 3), title="stringing boxes per image"); plt.tight_layout()

## Sequences for Model B

In [ ]:
import json
index = json.load(open("data/processed/sequences/index.json"))
seq = pd.DataFrame([{"id": r["id"], "kind": r["kind"], "split": r["split"], "frames": len(r["frames"]),
                     "stringing_frames": sum(1 for v in r["stringing"] if v == 1)} for r in index])
display(seq.groupby(["split", "kind"])["frames"].agg(["count", "median", "min", "max"]))

## Sample grid

In [ ]:
classes = sorted(trier["class"].unique())
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3 * len(classes)))
for i, c in enumerate(classes):
    files = list(Path("data/processed/trier_cls/train", c).glob("*.jpg"))
    for j, p in enumerate(random.sample(files, k=min(4, len(files)))):
        axes[i, j].imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        axes[i, j].set_title(f"{c}\n{p.name[:28]}", fontsize=8); axes[i, j].axis("off")
plt.tight_layout()

## Take-aways to check on the full dataset

* Are classes balanced per camera, or does one camera dominate a class (camera shortcut)?
* Are stringing and clean YOLO-Strings sequences visually different prints (scene shortcut for Model B)?
* Are box areas small enough that a higher detector `imgsz` is worth the cost?